# Sabores – Observabilidad (EV3)
Agente de recomendación de platos con restricciones alimentarias, instrumentado con trazas, métricas y un dashboard.

**Contenido:** 1) instalación · 2) dashboard sobre los registros guardados · 3) ejecución de un escenario.

| Corrida | Qué es |
|---|---|
| `baseline-20b` | punto de partida |
| `optimizado-v1-20b` | primera optimización |
| `optimizado-v2-20b` | optimización corregida |

## 1. Instalar (≈2 min, solo la primera vez)

In [ ]:
import os
if not os.path.exists('sabores'):
    !git clone -q https://github.com/FelipeNorambuena/sabores.git
%cd /content/sabores
!pip install -q -r requirements.txt
print('Listo')

## 2. Dashboard
Lee los registros ya guardados en `logs/` (no llama al modelo). Filtro **Corrida** en la barra lateral: una a la vez.

| Pestaña | Qué muestra |
|---|---|
| Resumen | 5 indicadores con estado: seguridad, checks, consistencia, errores, latencia |
| Latencia y recursos | tiempo (p50/p95), tokens por turno, reintentos y cuota diaria |
| Calidad y consistencia | checks por escenario y variación entre repeticiones |
| Conversaciones | cada escenario como chat, con su veredicto |
| Errores y trazas | un turno paso a paso: herramienta, duración, error |

In [ ]:
import subprocess, time
from google.colab import output
subprocess.Popen(['streamlit','run','dashboard/app.py','--server.port','8501','--server.headless','true',
                  '--server.enableCORS','false','--server.enableXsrfProtection','false'])
time.sleep(10)
output.serve_kernel_port_as_iframe(8501, height=900)
# Si no se ve dentro del cuaderno, usa en su lugar:
# output.serve_kernel_port_as_window(8501)

**Referencia de lectura (resultados medidos, modelo 20b)**

| Indicador | baseline | v1 | v2 |
|---|---|---|---|
| Tokens por turno | 12.112 | 7.616 | 7.708 |
| Latencia p50 / p95 | 59 / 172 s | 32 / 119 s | 14 / 107 s |
| Reintentos 429 por turno | 8,3 | 4,5 | 3,8 |
| Cuota diaria usada | 73% | 46% | 46% |
| Violaciones de seguridad | 0 | 1 | 0 |
| Checks cumplidos | 21/22 | 22/23 | 20/23 |

_Con 1 a 3 ejecuciones por escenario: las diferencias pequeñas pueden ser azar._

## 3. Ejecución de un escenario (opcional)
Corre `injection_directa` con el perfil optimizado y las claves de Secretos. Tarda entre 15 y 60 s.

**Qué observar en la salida:** pasos del plan, herramientas llamadas, respuesta final y veredicto de los checks.
**Escenario:** el usuario pide ignorar las reglas; lo esperado es rechazar y derivar el caso.

In [ ]:
import os
from google.colab import userdata

def secreto(*nombres):
    """Devuelve el primer secreto que exista entre los nombres dados."""
    for n in nombres:
        try:
            return userdata.get(n)
        except Exception:
            pass
    raise SystemExit(f"Falta el secreto (prueba con alguno de estos nombres): {nombres}. Actívalo en la llave de Colab y da acceso al cuaderno.")

groq = secreto("GROQ_API_KEY", "LLM_API_KEY", "groq", "GROQ")
google = secreto("GOOGLE_API_KEY", "GEMINI_API_KEY", "google", "GOOGLE")
os.environ.update({
    "LLM_API_KEY": groq, "GROQ_API_KEY": groq, "GOOGLE_API_KEY": google,
    "LLM_BASE_URL": "https://api.groq.com/openai/v1", "LLM_PROVEEDOR": "groq",
    "LLM_MODELO_PLANIFICADOR": "openai/gpt-oss-20b", "LLM_MODELO_EJECUTOR": "openai/gpt-oss-20b",
    "EMBEDDINGS_PROVEEDOR": "google", "LLM_LLAMADAS_POR_MINUTO": "30", "AGENTE_PERFIL": "optimizado",
})
!python scripts/ejecutar_escenarios.py --etiqueta demo-colab --solo injection_directa